# AI Data Engineering Project: BigQuery with Gemini Enterprise Agent Platform

Welcome to this beginner-friendly notebook exercise! In this project, you will learn how to bridge **Google BigQuery** and the **Gemini Enterprise Agent Platform** using the modern `google-genai` Python SDK.

We will query e-commerce analytics data from the public `thelook_ecommerce` dataset, pass contextual data into the Gemini platform, and leverage your configured enterprise assistant engine.

Originally, the following GitHub repository was referred to for this project:
https://github.com/nataindata/ai-data-engineering-project

In [1]:
# Install the unified Google Gen AI SDK and BigQuery dependencies
!pip install google-genai
!pip install google-cloud-bigquery
!pip install db-dtypes
!pip install pandas

---

#### ⚠️ Do not forget to "RESTART RUNTIME"

---

## Configuration and Client Initialization

Here, we initialize our project variables and authenticate the Gemini Enterprise Agent Platform client using your specific Project ID and Engine ID configurations.

⚠️ Make sure you set the parameters accordingly to that of your own project.

In [1]:
import os
from google import genai
from google.genai import types
from google.cloud import bigquery
import pandas as pd

In [2]:
# Define user project parameters
PROJECT_ID = "project-baf28c5d-7b6c-4d16-a41"
LOCATION = "global" # Discovery Engine agents typically reside on the 'global' regional control plane
ENGINE_ID = "gemini-enterprise-1791514593569"

In [3]:
# Initialize the Gemini Enterprise client
# Setting enterprise=True points the SDK to the Gemini Enterprise Agent Platform APIs
client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION
)

In [4]:
# Initialize BigQuery Client
bq_client = bigquery.Client(project=PROJECT_ID)

print("Clients initialized successfully for Gemini Enterprise Agent Platform.")

Clients initialized successfully for Gemini Enterprise Agent Platform.


## Extracting Data from BigQuery (`thelook_ecommerce`)

We will run a SQL query against the public `bigquery-public-data.thelook_ecommerce` dataset to extract summary metrics about order statuses and product categories. This summary will be fed into our Gemini assistant.

In [5]:
from google.colab import auth as google_auth

# Ensure user is authenticated to query BigQuery
google_auth.authenticate_user()

In [6]:
# Query summary metrics from thelook_ecommerce dataset
query = """
    SELECT
        status,
        COUNT(order_id) as total_orders,
        ROUND(SUM(sale_price), 2) as total_revenue
    FROM
        `bigquery-public-data.thelook_ecommerce.order_items`
    GROUP BY
        status
    ORDER BY
        total_revenue DESC
    LIMIT 5
"""

In [7]:
# Execute query and convert to Pandas DataFrame
df = bq_client.query(query).to_dataframe()

In [8]:
# Display the extracted data frame
print("Extracted Data Preview:")
display(df)

Extracted Data Preview:


,status,total_orders,total_revenue
0,Shipped,54082,3207167.91
1,Complete,45285,2707334.88
2,Processing,36511,2170184.42
3,Cancelled,26746,1588572.89
4,Returned,18218,1085462.90


In [9]:
# Convert dataframe details into a text context block for Gemini
data_context = df.to_string(index=False)

## Generating Insights with Gemini Enterprise Assistant

We will now send our processed BigQuery insights along with a prompt to your customized Gemini Enterprise Assistant engine (`gemini-enterprise-1791514593569`) to generate an executive summary and recommendations.

In [10]:
# Construct the prompt utilizing our BigQuery data context
prompt = f"""
You are an expert e-commerce data analyst assistant.
Here is a summary of recent order metrics extracted from the BigQuery 'thelook_ecommerce' dataset:

{data_context}

Please provide:
1. A brief executive breakdown of the revenue performance by order status.
2. Two actionable operational recommendations for an e-commerce store manager based on these numbers.
"""

In [12]:
success = False

try:
    from google.api_core.client_options import ClientOptions
    import google.cloud.discoveryengine_v1 as discoveryengine

    # Try potential regions where the Discovery Engine application might be deployed
    potential_locations = ["us", "global", "eu"]

    for loc in potential_locations:
        client_options = (
            ClientOptions(api_endpoint=f"{loc}-discoveryengine.googleapis.com")
            if loc != "global"
            else None
        )

        try:
            engine_client = discoveryengine.ConversationalSearchServiceClient(client_options=client_options)

            # Try engine path first
            engine_path = f"projects/{PROJECT_ID}/locations/{loc}/collections/default_collection/engines/{ENGINE_ID}"
            request = discoveryengine.ConverseConversationRequest(
                name=f"{engine_path}/conversations/-",
                query=discoveryengine.TextInput(input=prompt),
            )
            response = engine_client.converse_conversation(request=request)
            print(f"--- Gemini Enterprise Assistant Analysis ({loc} engine) ---")
            print(response.reply.reply)
            success = True
            break
        except Exception as e_engine:
            try:
                # Try raw data store fallback within that region
                fallback_path = f"projects/{PROJECT_ID}/locations/{loc}/collections/default_collection/dataStores/{ENGINE_ID}"
                request = discoveryengine.ConverseConversationRequest(
                    name=f"{fallback_path}/conversations/-",
                    query=discoveryengine.TextInput(input=prompt),
                )
                response = engine_client.converse_conversation(request=request)
                print(f"--- Gemini Enterprise Assistant Analysis ({loc} dataStore fallback) ---")
                print(response.reply.reply)
                success = True
                break
            except Exception as e_ds:
                continue
except ModuleNotFoundError:
    print("google-cloud-discoveryengine library not installed. Skipping Enterprise Agent path.")

if not success:
    print(f"Falling back to Vertex AI model for analysis...")
    try:
        # Fallback to Vertex AI routing using GCP IAM credentials (avoids 'No API key' error)
        from google import genai
        fallback_client = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location="us-central1"
        )
        response = fallback_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
        )
        print("--- Gemini Vertex AI Assistant Analysis (Fallback) ---")
        print(response.text)
    except Exception as fallback_error:
        print(f"All retrieval paths failed. Error: {fallback_error}")

google-cloud-discoveryengine library not installed. Skipping Enterprise Agent path.
Falling back to Vertex AI model for analysis...
--- Gemini Vertex AI Assistant Analysis (Fallback) ---
Here's an executive breakdown and operational recommendations based on the provided order metrics:

---

### 1. Executive Breakdown of Revenue Performance by Order Status

The e-commerce store is generating substantial revenue, with successful orders (Shipped and Complete) accounting for nearly **\$5.9 million**, representing approximately 55% of the total revenue across all status categories. An additional **\$2.17 million** is currently in the 'Processing' pipeline, indicating healthy ongoing sales activity.

However, a significant portion of potential revenue, totaling over **\$2.6 million**, is being lost due to 'Cancelled' (\$1.59M) and 'Returned' (\$1.09M) orders. These problematic statuses together represent almost **25% of the total revenue across all statuses**, highlighting a critical area fo